In [26]:
contenu = """GMAIL_EXPEDITEUR=raadha.2910@gmail.com
GMAIL_MOT_DE_PASSE=kzfd msgr ykwg duse
GMAIL_DESTINATAIRE=shivadheepi@gmail.com
GROQ_API_KEY=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
"""

with open(".env", "w") as f:
    f.write(contenu)

print("Fichier .env créé !")

Fichier .env créé !


In [24]:
import os
print(os.path.exists(".env"))
print(os.getcwd())

True
/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/02_Notebooks_Python


In [25]:
with open(".env", "r") as f:
    print(f.read())

GMAIL_EXPEDITEUR=raadha.2910@gmail.com
GMAIL_MOT_DE_PASSE=kzfd msgr ykwg duse
GMAIL_DESTINATAIRE=shivadheepi@gmail.com
GROQ_API_KEY=colle_ta_clé_groq_ici



In [12]:
import requests
import pandas as pd
import re
import time  # 👈 AJOUTÉ

SHOP_URL = "https://nesridiscount.com"

# ============================================================
# 1. RÉCUPÉRER TOUS LES PRODUITS (pagination avec pause)
# ============================================================

tous_les_produits = []
page = 1

while True:
    url = f"{SHOP_URL}/products.json?limit=250&page={page}"
    response = requests.get(url)
    
    if response.status_code == 429:
        print(f"⏳ Rate limit atteint — pause 5 secondes...")
        time.sleep(5)  # 👈 On attend et on réessaie
        continue
    
    if response.status_code != 200:
        print(f"❌ Erreur inattendue : {response.status_code}")
        break
    
    produits_page = response.json().get("products", [])
    
    if not produits_page:
        break
    
    tous_les_produits.extend(produits_page)
    print(f"Page {page} → {len(produits_page)} produits récupérés")
    page += 1
    time.sleep(1)  # 👈 Pause 1 seconde entre chaque page (évite le 429)

print(f"\n✅ Total récupéré : {len(tous_les_produits)} produits")

Page 1 → 250 produits récupérés
Page 2 → 250 produits récupérés
Page 3 → 250 produits récupérés
Page 4 → 250 produits récupérés
Page 5 → 250 produits récupérés
Page 6 → 250 produits récupérés
Page 7 → 250 produits récupérés
Page 8 → 250 produits récupérés
Page 9 → 250 produits récupérés
Page 10 → 250 produits récupérés
Page 11 → 250 produits récupérés
Page 12 → 250 produits récupérés
Page 13 → 250 produits récupérés
Page 14 → 250 produits récupérés
Page 15 → 250 produits récupérés
Page 16 → 250 produits récupérés
Page 17 → 250 produits récupérés
Page 18 → 250 produits récupérés
Page 19 → 250 produits récupérés
Page 20 → 32 produits récupérés

✅ Total récupéré : 4782 produits


In [14]:
# ============================================================
# 2. TRANSFORMER EN DATAFRAME PROPRE
# ============================================================

def clean_html(text):
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:600]

lignes = []

for produit in tous_les_produits:
    # On prend uniquement la première variante
    variante = produit["variants"][0] if produit["variants"] else {}
    image    = produit["images"][0]["src"] if produit["images"] else ""
    
    prix_vente = float(variante.get("price") or 0)
    prix_barre = variante.get("compare_at_price")
    prix_barre = float(prix_barre) if prix_barre else None
    
    # Ignorer les produits sans prix
    if prix_vente <= 0:
        continue
    
    economie   = round(prix_barre - prix_vente, 2) if prix_barre else None
    remise_pct = round((economie / prix_barre) * 100) if prix_barre else None
    
    lignes.append({
        "Handle"     : produit.get("handle", ""),
        "Titre"      : produit.get("title", ""),
        "Vendeur"    : produit.get("vendor", ""),
        "Prix_vente" : prix_vente,
        "Prix_barre" : prix_barre,
        "Economie"   : economie,
        "Remise_pct" : remise_pct,
        "Description": clean_html(produit.get("body_html", "")),
        "Image"      : image,
        "Type"       : produit.get("product_type", ""),
    })

df = pd.DataFrame(lignes)

# Supprimer les remises aberrantes (>70%)
nb_avant = len(df)
df = df[~(df["Remise_pct"] > 70)].copy()
df = df.reset_index(drop=True)

print(f"{'='*50}")
print(f"✅ CATALOGUE FINAL : {len(df)} produits propres")
print(f"   → {df['Prix_barre'].notna().sum()} produits EN PROMO")
print(f"   → {df['Prix_barre'].isna().sum()} produits à prix fixe")
print(f"   → Remises aberrantes supprimées : {nb_avant - len(df)}")
print(f"{'='*50}")
print(df[["Titre", "Prix_vente", "Prix_barre", "Remise_pct"]].head(10))

✅ CATALOGUE FINAL : 4778 produits propres
   → 2560 produits EN PROMO
   → 2218 produits à prix fixe
   → Remises aberrantes supprimées : 3
                                               Titre  Prix_vente  Prix_barre  \
0  Brûleur 28 mbar d'origine Campingaz Plancha RL...        25.0         NaN   
1  Kit électrodes avec support et vis d'origine C...        19.0         NaN   
2  Entonnoir graisse d'origine Campingaz Plancha ...        19.0         NaN   
3  Tubulure gaz 28 mbar 1/2" (FR) d'origine Campi...        64.0         NaN   
4  Pare chaleur d'origine Campingaz Plancha EX3 –...        24.0         NaN   
5     Cuve d'origine Campingaz Plancha EX3 – 2217656        49.0         NaN   
6  Rail télescopique 2 niveaux pour four SIEMENS ...        66.5       133.0   
7  Plaque de cuisson en fonte émaillée d'origine ...       109.0         NaN   
8  Console noire d'origine Campingaz Plancha EX3 ...        26.0         NaN   
9  Couvercle noir complet avec poignée, biellette...        

In [16]:
# ============================================================
# 3. VÉRIFICATION — Tester un produit en promo
# ============================================================

# Prendre le premier produit en promo
produit_test = df[df["Prix_barre"].notna()].iloc[0]

print("🔍 VÉRIFICATION PRODUIT TEST")
print("="*50)
print(f"Titre      : {produit_test['Titre']}")
print(f"Vendeur    : {produit_test['Vendeur']}")
print(f"Prix vente : {produit_test['Prix_vente']} €")
print(f"Prix barré : {produit_test['Prix_barre']} €")
print(f"Remise     : -{int(produit_test['Remise_pct'])}%")
print(f"Économie   : {produit_test['Economie']} €")
print(f"Image      : {produit_test['Image']}")
print("="*50)
print(f"\n👉 Va vérifier ce prix sur le site :")
print(f"   https://nesridiscount.com/products/{produit_test['Handle']}")

🔍 VÉRIFICATION PRODUIT TEST
Titre      : Rail télescopique 2 niveaux pour four SIEMENS HZ638270EM
Vendeur    : Siemens
Prix vente : 66.5 €
Prix barré : 133.0 €
Remise     : -50%
Économie   : 66.5 €
Image      : https://cdn.shopify.com/s/files/1/0982/1428/1557/files/Railtelescopique2niveauxpourfourSIEMENSHZ638270.png?v=1770374856

👉 Va vérifier ce prix sur le site :
   https://nesridiscount.com/products/rail-telescopique-2-niveaux-pour-four-siemens-hz638270-copie


In [17]:
# ============================================================
# 4. SÉLECTION DES 4 PRODUITS DU JOUR (promo + saison été)
# ============================================================

import os

chemin_compteur = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/03_Donnees_propres/compteur_rotation.txt"

# Mots-clés produits d'été
mots_cles_ete = [
    "ventilateur", "climatiseur", "climatisation", "rafraîchisseur", "rafraichisseur",
    "barbecue", "plancha", "glacière", "glaciere", "congélateur", "congelateur",
    "frigo", "réfrigérateur", "refrigerateur", "camping", "portabl",
    "piscine", "jardin", "terrasse", "extérieur", "exterieur"
]

def est_produit_ete(prod):
    texte = (str(prod['Titre']) + " " + str(prod['Description']) + " " + str(prod['Type'])).lower()
    return any(mot in texte for mot in mots_cles_ete)

# Créer les catégories
df['Est_ete']   = df.apply(est_produit_ete, axis=1)
df['Est_promo'] = df['Prix_barre'].notna()

df_promo_ete   = df[df['Est_promo'] & df['Est_ete']].copy()    # 🔥 Promo + Été
df_promo       = df[df['Est_promo'] & ~df['Est_ete']].copy()   # Promo seulement
df_ete         = df[~df['Est_promo'] & df['Est_ete']].copy()   # Été seulement

print(f"📊 Répartition du catalogue :")
print(f"   🔥 Promo + Été  : {len(df_promo_ete)} produits")
print(f"   💰 Promo seul   : {len(df_promo)} produits")
print(f"   ☀️  Été seul     : {len(df_ete)} produits")

# Lire le compteur
if not os.path.exists(chemin_compteur):
    index_hier = -1
    print("\nPremier démarrage — compteur initialisé à 0")
else:
    with open(chemin_compteur, 'r') as f:
        index_hier = int(f.read().strip())
    print(f"\nDernier index envoyé : {index_hier}")

# Sélectionner 4 produits en priorité : promo+été → promo → été
pool_prioritaire = pd.concat([df_promo_ete, df_promo, df_ete], ignore_index=True)
pool_prioritaire = pool_prioritaire.drop_duplicates(subset='Handle').reset_index(drop=True)

indices_aujourdhui = [(index_hier + i + 1) % len(pool_prioritaire) for i in range(4)]
liste_produits     = [pool_prioritaire.iloc[idx].copy() for idx in indices_aujourdhui]

# Nettoyage vendeur
for prod in liste_produits:
    if pd.isna(prod['Vendeur']) or str(prod['Vendeur']).strip() == '':
        prod['Vendeur'] = "Nesri Discount"

print(f"\n{'='*50}")
print(f"📦 4 PRODUITS DU JOUR (promo + saison été)")
print(f"{'='*50}")
for i, prod in enumerate(liste_produits, 1):
    tag = "🔥 PROMO+ÉTÉ" if prod['Est_promo'] and prod['Est_ete'] else "💰 PROMO" if prod['Est_promo'] else "☀️ ÉTÉ"
    if pd.notna(prod['Prix_barre']):
        prix_str = f"{int(prod['Prix_barre'])} € → {int(prod['Prix_vente'])} € (-{int(prod['Remise_pct'])}%)"
    else:
        prix_str = f"{int(prod['Prix_vente'])} €"
    print(f"[{i}/4] {tag} — {prod['Titre'][:45]}...")
    print(f"       💰 {prix_str}")
print(f"{'='*50}")

📊 Répartition du catalogue :
   🔥 Promo + Été  : 623 produits
   💰 Promo seul   : 1937 produits
   ☀️  Été seul     : 628 produits

Dernier index envoyé : 48

📦 4 PRODUITS DU JOUR (promo + saison été)
[1/4] 🔥 PROMO+ÉTÉ — Réchaud gaz à paella RECPRO50...
       💰 95 € → 75 € (-21%)
[2/4] 🔥 PROMO+ÉTÉ — Chauffage Tour céramique mobile 2000 W DREXON...
       💰 85 € → 75 € (-12%)
[3/4] 🔥 PROMO+ÉTÉ — Chauffage Radiant d'extérieur 1300 W DREXON 7...
       💰 115 € → 90 € (-22%)
[4/4] 🔥 PROMO+ÉTÉ — Pack de 24 Cartouches de gaz CAMPINGAZ CV 470...
       💰 215 € → 190 € (-12%)


In [19]:
# ============================================================
# 5. RÉSUMÉ DES 4 PRODUITS DU JOUR
# ============================================================

print(f"📊 RÉSUMÉ DU CATALOGUE EN DIRECT :")
print(f"   → {len(df)} produits récupérés depuis nesridiscount.com")
print(f"   → {df['Prix_barre'].notna().sum()} en promo / {df['Prix_barre'].isna().sum()} à prix fixe")
print(f"   → Source : temps réel (API Shopify publique) ✅")
print(f"\n📦 4 produits sélectionnés aujourd'hui :")
for i, prod in enumerate(liste_produits, 1):
    print(f"   [{i}] {prod['Titre'][:55]}...")

📊 RÉSUMÉ DU CATALOGUE EN DIRECT :
   → 4778 produits récupérés depuis nesridiscount.com
   → 2560 en promo / 2218 à prix fixe
   → Source : temps réel (API Shopify publique) ✅

📦 4 produits sélectionnés aujourd'hui :
   [1] Réchaud gaz à paella RECPRO50...
   [2] Chauffage Tour céramique mobile 2000 W DREXON 931500...
   [3] Chauffage Radiant d'extérieur 1300 W DREXON 705500...
   [4] Pack de 24 Cartouches de gaz CAMPINGAZ CV 470 PLUS...


In [20]:
# ============================================================
# 6. GÉNÉRATION DES 4 POSTS INSTAGRAM VIA IA
# ============================================================

from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def format_prix(prix):
    if pd.isna(prix):
        return ""
    return f"{int(prix)} €"

def generer_post(prod):
    if pd.notna(prod['Prix_barre']):
        info_prix = (
            f"Prix original : {format_prix(prod['Prix_barre'])}\n"
            f"Prix actuel   : {format_prix(prod['Prix_vente'])}\n"
            f"Économie      : {format_prix(prod['Economie'])} soit -{int(prod['Remise_pct'])}% de remise"
        )
    else:
        info_prix = f"Prix : {format_prix(prod['Prix_vente'])}"

    prompt = f"""Tu es un expert en marketing digital pour une boutique en ligne française d'électroménager discount appelée Nesri Discount.

Génère un post Instagram accrocheur pour le produit suivant :

Nom du produit : {prod['Titre']}
Marque : {prod['Vendeur']}
{info_prix}
Description : {prod['Description'][:400]}

Règles importantes :
- Commence par une phrase d'accroche percutante avec des emojis
- Mets en valeur le prix et l'économie si le produit est en promo
- Les hashtags doivent être sans accents, pertinents au produit et à la marque, maximum 8
- Le ton doit être enthousiaste et dynamique comme un vrai community manager
- Le post doit faire entre 6 et 10 lignes avec des sauts de ligne entre chaque idée
- Écris en français
- Termine toujours par : 🛒 Disponible sur NesriDiscount.fr
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.8
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"🔥 Offre exceptionnelle sur {prod['Titre']} ! 🛒 Disponible sur NesriDiscount.fr"

# Générer les 4 posts
print("🤖 Génération des 4 posts Instagram...")
print("="*50)

posts_generes = []
for i, prod in enumerate(liste_produits, 1):
    print(f"\n📱 POST [{i}/4] — {prod['Titre'][:50]}...")
    post = generer_post(prod)
    posts_generes.append(post)
    print(post)
    print("-"*50)

print(f"\n✅ {len(posts_generes)} posts générés !")

🤖 Génération des 4 posts Instagram...

📱 POST [1/4] — Réchaud gaz à paella RECPRO50...
🔥🍲 Prêts à cuisiner comme des pros avec notre réchaud gaz à paella RECPRO50 ! 

Ce réchaud est équipé de 2 robinets qui contrôlent indépendamment 3 rampes, pour une cuisson personnalisée à votre goût.

Actuellement en promo à 75 €, vous bénéficiez d’une économie de 20 € par rapport au prix original de 95 €, soit -21% de remise !

C’est l’occasion idéale de réaliser des paellas délicieuses avec une puissance accrue et une répartition de chaleur optimale.

#NesriDiscount #RechaudGaz #Paella #Cuisine #Electromenager #Discount #Promo #Cuisson

🛒 Disponible sur NesriDiscount.fr
--------------------------------------------------

📱 POST [2/4] — Chauffage Tour céramique mobile 2000 W DREXON 9315...
🔥 Chauffez votre maison sans vous ruiner ! 💸
Nous avons une offre incroyable sur le Chauffage Tour céramique mobile 2000 W DREXON 931500, avec un prix qui passe de 85 € à 75 €, ce qui vous permet d'économiser 10 

In [21]:
# ============================================================
# 7. GÉNÉRATION DES 4 IMAGES EN MÉMOIRE
# ============================================================

from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import requests as req

COLOR_BG_TOP    = (12, 22, 54, 255)
COLOR_BG_BOTTOM = (20, 32, 68, 255)
COLOR_WHITE     = (255, 255, 255, 255)
COLOR_MUTED     = (170, 185, 220, 255)
COLOR_ACCENT    = (0, 245, 156, 255)
COLOR_RED       = (227, 6, 19, 255)

def load_font(size, bold=False):
    candidates = [
        "/Library/Fonts/Arial Bold.ttf" if bold else "/Library/Fonts/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

def text_width(draw, text, font):
    bbox = draw.textbbox((0, 0), text, font=font)
    return bbox[2] - bbox[0]

def fit_font(draw, text, max_width, start=40, min_size=22, bold=True):
    for size in range(start, min_size - 1, -2):
        font = load_font(size, bold=bold)
        if text_width(draw, text, font) <= max_width:
            return font
    return load_font(min_size, bold=bold)

def wrap_text(draw, text, font, max_width, max_lines=2):
    words = text.split()
    lines, current = [], ""
    for w in words:
        test = w if not current else current + " " + w
        if text_width(draw, test, font) <= max_width:
            current = test
        else:
            if current:
                lines.append(current)
            current = w
    if current:
        lines.append(current)
    return "\n".join(lines[:max_lines])

def generer_image_en_memoire(prod):
    W, H = 1080, 1350
    canvas = Image.new("RGBA", (W, H))
    draw   = ImageDraw.Draw(canvas)

    for y in range(H):
        t = y / H
        r = int(COLOR_BG_TOP[0] + (COLOR_BG_BOTTOM[0] - COLOR_BG_TOP[0]) * t)
        g = int(COLOR_BG_TOP[1] + (COLOR_BG_BOTTOM[1] - COLOR_BG_TOP[1]) * t)
        b = int(COLOR_BG_TOP[2] + (COLOR_BG_BOTTOM[2] - COLOR_BG_TOP[2]) * t)
        draw.line([(0, y), (W, y)], fill=(r, g, b, 255))

    draw.rectangle([40, 40, W-40, H-40], outline=(255, 255, 255, 15), width=2)

    bandeau = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    bd = ImageDraw.Draw(bandeau)
    bd.polygon([(W-350, 0), (W, 350), (W, 230), (W-230, 0)], fill=COLOR_RED)
    canvas = Image.alpha_composite(canvas, bandeau)
    draw   = ImageDraw.Draw(canvas)
    draw.text((W-195, 75),  "OFFRE", font=load_font(24, bold=True), fill=COLOR_WHITE)
    draw.text((W-185, 110), "FLASH", font=load_font(24, bold=True), fill=COLOR_WHITE)

    logo_path = "/Users/surenthinisivakumar/Documents/Stage_Nesri_Discount/logo_nesri.png"
    if os.path.exists(logo_path):
        logo   = Image.open(logo_path).convert("RGBA")
        logo_w = 300
        logo_h = int(logo.height * (logo_w / logo.width))
        logo   = logo.resize((logo_w, logo_h))
        canvas.alpha_composite(logo, (60, 60))

    image_url = prod.get("Image", "")
    if pd.notna(image_url) and str(image_url).strip() != "":
        try:
            res      = req.get(image_url, timeout=10)
            img_prod = Image.open(BytesIO(res.content)).convert("RGBA")
            h_box    = int(W * 0.55)
            img_prod.thumbnail((h_box, h_box))
            px = (W - img_prod.width) // 2
            py = int(H * 0.18)
            canvas.alpha_composite(img_prod, (px, py))
        except Exception as e:
            print(f"⚠️ Image produit non chargée : {e}")

    text_y = int(H * 0.60)
    draw.text((60, text_y), str(prod['Vendeur']).upper(),
              font=load_font(24, bold=True), fill=COLOR_MUTED)

    titre_font    = fit_font(draw, prod['Titre'], W-120, start=38, min_size=22, bold=True)
    titre_wrapped = wrap_text(draw, prod['Titre'], titre_font, W-120)
    draw.multiline_text((60, text_y+40), titre_wrapped,
                        font=titre_font, fill=COLOR_WHITE, spacing=6)

    bbox_titre = draw.multiline_textbbox((60, text_y+40), titre_wrapped,
                                          font=titre_font, spacing=6)
    prix_y = bbox_titre[3] + 40

    txt_pv = f"{int(prod['Prix_vente'])} €"
    draw.text((60, prix_y), txt_pv, font=load_font(85, bold=True), fill=COLOR_WHITE)

    if pd.notna(prod['Prix_barre']):
        larg_pv = text_width(draw, txt_pv, load_font(85, bold=True))
        txt_pb  = f"{int(prod['Prix_barre'])} €"
        next_x  = 60 + larg_pv + 30
        draw.text((next_x, prix_y+35), txt_pb, font=load_font(36), fill=COLOR_MUTED)
        larg_pb = text_width(draw, txt_pb, load_font(36))
        draw.line([(next_x, prix_y+53), (next_x+larg_pb, prix_y+53)], fill=COLOR_RED, width=3)
        draw.rounded_rectangle([W-240, prix_y+10, W-60, prix_y+75], radius=14, fill=COLOR_ACCENT)
        txt_rem = f"-{int(prod['Remise_pct'])}%"
        tw_rem  = text_width(draw, txt_rem, load_font(32, bold=True))
        draw.text((W-150-(tw_rem//2), prix_y+22), txt_rem,
                  font=load_font(32, bold=True), fill=(12, 22, 54, 255))

    draw.rounded_rectangle([60, H-140, W-60, H-60], radius=22, fill=COLOR_RED)
    cta    = "PROFITER DE L'OFFRE SUR LE SITE  ➔"
    tw_cta = text_width(draw, cta, load_font(28, bold=True))
    draw.text(((W-tw_cta)//2, H-108), cta, font=load_font(28, bold=True), fill=COLOR_WHITE)

    buffer = BytesIO()
    canvas.convert("RGB").save(buffer, format="PNG")
    buffer.seek(0)
    return buffer.read()

# Générer les 4 images
print("🎨 Génération des 4 images en mémoire...")
images_bytes = []
for i, prod in enumerate(liste_produits, 1):
    img = generer_image_en_memoire(prod)
    images_bytes.append(img)
    print(f"   ✅ Image [{i}/4] générée ({len(img) // 1024} Ko)")

print(f"\n✅ {len(images_bytes)} images générées en mémoire !")

🎨 Génération des 4 images en mémoire...
   ✅ Image [1/4] générée (188 Ko)
   ✅ Image [2/4] générée (107 Ko)
   ✅ Image [3/4] générée (238 Ko)
   ✅ Image [4/4] générée (288 Ko)

✅ 4 images générées en mémoire !


In [22]:
# ============================================================
# 8. ENVOI PAR MAIL — 4 PRODUITS + 4 IMAGES EN PIÈCES JOINTES
# ============================================================

import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

expediteur   = "raadha.2910@gmail.com"
mot_de_passe = "kzfd msgr ykwg duse"
destinataire = "shivadheepi@gmail.com"

# Construire les blocs HTML pour les 4 produits
blocs_html = ""
for i, (prod, post) in enumerate(zip(liste_produits, posts_generes), 1):
    if pd.notna(prod['Prix_barre']):
        badge     = f'<span style="background:#E30613;color:#fff;font-size:12px;font-weight:bold;padding:3px 8px;border-radius:4px;margin-left:8px;">-{int(prod["Remise_pct"])}%</span>'
        prix_barre_html = f'<span style="color:#999;text-decoration:line-through;margin-right:8px;font-size:14px;">{int(prod["Prix_barre"])} €</span>'
    else:
        badge           = ""
        prix_barre_html = ""

    blocs_html += f"""
    <div style="border:1px solid #eee;border-radius:8px;padding:16px;margin-bottom:20px;background:#fafafa;">
      <table width="100%" cellspacing="0" cellpadding="0">
        <tr>
          <td width="30%" align="center">
            <img src="{prod['Image']}" style="max-width:120px;max-height:120px;object-fit:contain;" />
          </td>
          <td width="70%" style="padding-left:15px;">
            <p style="margin:0 0 4px 0;font-size:11px;color:#888;">PRODUIT {i}/4</p>
            <h3 style="margin:0 0 6px 0;color:#1a1a1a;font-size:15px;">{prod['Titre']}</h3>
            <p style="margin:0 0 8px 0;color:#E30613;font-weight:bold;font-size:18px;">
              {prix_barre_html}{int(prod['Prix_vente'])} € {badge}
            </p>
            <p style="margin:0 0 6px 0;font-size:11px;color:#888;">📎 Visuel Instagram : post_instagram_{i}.png</p>
            <p style="margin:0;font-size:12px;color:#444;line-height:1.5;
                      white-space:pre-line;font-style:italic;">{post}</p>
          </td>
        </tr>
      </table>
    </div>
    """

html = f"""
<!DOCTYPE html>
<html>
<body style="margin:0;padding:0;background:#f4f4f4;font-family:Arial,sans-serif;">
<div style="max-width:600px;margin:20px auto;background:#fff;border-radius:12px;
            overflow:hidden;box-shadow:0 4px 10px rgba(0,0,0,0.1);">

  <div style="background:#0c1636;padding:24px;text-align:center;">
    <h1 style="color:#fff;font-size:26px;margin:0;letter-spacing:1px;">NESRI DISCOUNT</h1>
    <p style="color:#00f59c;font-size:14px;margin:5px 0 0;font-weight:bold;">
      🔥 SÉLECTION DU JOUR — 4 PRODUITS
    </p>
  </div>

  <div style="padding:20px 25px;">
    <p style="font-size:15px;color:#333;">Bonjour équipe communication,</p>
    <p style="font-size:14px;color:#666;margin-bottom:20px;">
      Voici votre sélection automatisée de 4 produits du jour avec leurs posts Instagram
      et visuels prêts à publier (pièces jointes).
    </p>

    {blocs_html}

    <div style="text-align:center;margin-top:25px;">
      <a href="https://nesridiscount.fr"
         style="background:#E30613;color:#fff;text-decoration:none;
                padding:12px 24px;border-radius:6px;font-weight:bold;display:inline-block;">
        Accéder au site officiel
      </a>
    </div>
  </div>

</div>
</body>
</html>
"""

# Construire le mail
message            = MIMEMultipart("mixed")
message["From"]    = expediteur
message["To"]      = destinataire
message["Subject"] = "🔥 Sélection du jour — 4 produits Nesri Discount"

# Corps HTML
msg_html = MIMEMultipart("alternative")
msg_html.attach(MIMEText(html, "html"))
message.attach(msg_html)

# 4 images en pièces jointes
for i, img_bytes in enumerate(images_bytes, 1):
    piece = MIMEBase("application", "octet-stream")
    piece.set_payload(img_bytes)
    encoders.encode_base64(piece)
    piece.add_header("Content-Disposition", f'attachment; filename="post_instagram_{i}.png"')
    message.attach(piece)

# Envoi
try:
    print("🚀 Envoi du mail avec 4 produits + 4 images...")
    with smtplib.SMTP("smtp.gmail.com", 587) as serveur:
        serveur.ehlo()
        serveur.starttls()
        serveur.ehlo()
        serveur.login(expediteur, mot_de_passe)
        serveur.sendmail(expediteur, destinataire, message.as_string())
    print("🎉 Mail envoyé avec succès !")

    # Mettre à jour le compteur sur le 4ème produit
    with open(chemin_compteur, 'w') as f:
        f.write(str(indices_aujourdhui[-1]))
    print(f"✅ Compteur mis à jour → demain on repart à l'index {indices_aujourdhui[-1] + 1}")

except Exception as e:
    print(f"🚨 Erreur lors de l'envoi : {e}")

🚀 Envoi du mail avec 4 produits + 4 images...
🎉 Mail envoyé avec succès !
✅ Compteur mis à jour → demain on repart à l'index 53
